# Improved Core PCE Nowcast from CPI and PPI

This standalone notebook estimates core PCE before the official release using the
full detailed BEA PCE partition. It starts from the economically transparent
component model in `core_pce_nowcast_detailed.ipynb` and adds accuracy improvements
that are feasible in real time:

1. exact signed coverage of core PCE using the finest stable BEA partition;
2. CPI/PPI component proxies plus a causal six-month BEA fallback;
3. **adaptive component shrinkage** toward the fallback, with weights learned only
   from component errors available before each forecast month;
4. an **online ensemble of two hospital deflators**, addressing the largest mapped
   source inconsistency in the original model;
5. a small **rolling aggregate bias correction**, again using only prior errors;
6. pseudo-out-of-sample-style scoring, empirical uncertainty bands, contribution
   diagnostics, release-quality checks, and credentials loaded outside the notebook.

The raw component forecast is always retained as a benchmark. The improved forecast
is accepted only to the extent that it beats the raw model after its 24-month warm-up.


In [ ]:
import json
import os
import time
import urllib.parse
import urllib.request
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

BASE_DIR = Path.cwd()
DETAIL_FILE = BASE_DIR / "Detailed PCE.xlsx"
OUTPUT_DIR = BASE_DIR / "outputs"
API_KEYS_FILE = BASE_DIR / "API Keys.txt"


def load_api_key(name):
    """Load NAME_API_KEY from the environment, then from API Keys.txt."""
    env_name = f"{name.upper()}_API_KEY"
    if os.getenv(env_name):
        return os.environ[env_name].strip()
    if API_KEYS_FILE.exists():
        for raw_line in API_KEYS_FILE.read_text(encoding="utf-8-sig").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            sep = ":" if ":" in line else "=" if "=" in line else None
            if sep:
                key, value = line.split(sep, 1)
                if key.strip().upper() == name.upper() and value.strip():
                    return value.strip()
    raise RuntimeError(
        f"Missing {env_name}. Set it in the environment or add "
        f"'{name.upper()}: <key>' to {API_KEYS_FILE.name}."
    )


BLS_API_KEY = load_api_key("BLS")
FRED_API_KEY = load_api_key("FRED")
BEA_API_KEY = load_api_key("BEA")

BACKTEST_START = pd.Timestamp("2015-01-01")
EVALUATION_START = pd.Timestamp("2017-01-01")  # 24-month adaptive warm-up
SOURCE_START_YEAR = 2009
BEA_START_YEAR = 2013

# Chosen before the final scoring cell and always estimated with data through t-1.
FALLBACK_WINDOW = 6
ADAPT_WINDOW = 120
ADAPT_MIN_OBS = 24
HOSPITAL_ENSEMBLE_WINDOW = 36
BIAS_WINDOW = 60
BIAS_MIN_OBS = 24
HOSPITAL_ALT_SERIES = "PCU622110622110"


## 1. Parse the detailed PCE hierarchy


In [ ]:
det = pd.read_excel(DETAIL_FILE, usecols=[0, 1, 2], dtype={1: str})
det.columns = ["line", "desc", "code"]
det = det.dropna(subset=["line"]).copy()
det["line"] = det["line"].astype(int)
det["indent"] = det["desc"].str.len() - det["desc"].str.lstrip().str.len()
det["desc"] = det["desc"].str.strip()
det = det.set_index("line").sort_index()
det.loc[1, "indent"] = -1  # title row is center-padded in the workbook
descriptions = det["desc"]

parent = {}
stack = []
for L, row in det.iterrows():
    while stack and stack[-1][1] >= row["indent"]:
        stack.pop()
    parent[L] = stack[-1][0] if stack else None
    stack.append((L, row["indent"]))
children = {}
for L, p in parent.items():
    children.setdefault(p, []).append(L)


def ancestors(L):
    out = []
    p = parent.get(L)
    while p is not None:
        out.append(p)
        p = parent.get(p)
    return out


def descendants(L):
    out = set()
    for k in children.get(L, []):
        out.add(k)
        out |= descendants(k)
    return out


sign = {L: (-1) ** sum(descriptions[a].startswith("Less:")
                       for a in [L] + ancestors(L)) for L in det.index}
print(f"{len(det)} lines, {sum(1 for s in sign.values() if s < 0)} with negative sign")

## 2. Data-fetch helpers


In [ ]:
def bls_fetch(series_ids, start_year, end_year):
    """Fetch monthly BLS series (chunks of 50). Returns wide DataFrame + missing ids."""
    frames = {}
    series_ids = list(series_ids)
    for i in range(0, len(series_ids), 50):
        chunk = series_ids[i:i + 50]
        payload = json.dumps({
            "seriesid": chunk,
            "startyear": str(start_year),
            "endyear": str(end_year),
            "registrationkey": BLS_API_KEY,
        }).encode()
        req = urllib.request.Request(
            "https://api.bls.gov/publicAPI/v2/timeseries/data/",
            data=payload, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=120) as resp:
            out = json.loads(resp.read())
        if out.get("status") != "REQUEST_SUCCEEDED":
            raise RuntimeError(f"BLS API: {out.get('status')} {out.get('message')}")
        for s in out["Results"]["series"]:
            rows = {
                pd.Timestamp(int(r["year"]), int(r["period"][1:]), 1): float(r["value"])
                for r in s["data"]
                if r["period"].startswith("M") and r["period"] != "M13"
                and r["value"] not in ("-", "")
            }
            if rows:
                frames[s["seriesID"]] = pd.Series(rows)
        time.sleep(0.2)
    df = pd.DataFrame(frames).sort_index()
    missing = [sid for sid in series_ids if sid not in df.columns]
    return df, missing


def fred_fetch(series_id, start="2009-01-01"):
    """Fetch one FRED series as a monthly Series."""
    url = ("https://api.stlouisfed.org/fred/series/observations"
           f"?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json"
           f"&observation_start={start}")
    with urllib.request.urlopen(url, timeout=60) as resp:
        obs = json.loads(resp.read())["observations"]
    return pd.Series({pd.Timestamp(o["date"]): float(o["value"])
                      for o in obs if o["value"] != "."})

## 3. Pull BEA underlying detail tables


In [ ]:
BEA_TABLES = {"prices": "U20404", "nominal": "U20405", "real": "U20406"}
matrices = {}

for label, table_name in BEA_TABLES.items():
    rows = []
    for year in range(BEA_START_YEAR, date.today().year + 1):
        params = {
            "UserID": BEA_API_KEY, "method": "GetData",
            "DataSetName": "NIUnderlyingDetail", "TableName": table_name,
            "Frequency": "M", "Year": str(year), "ResultFormat": "JSON",
        }
        url = "https://apps.bea.gov/api/data?" + urllib.parse.urlencode(params)
        with urllib.request.urlopen(url, timeout=120) as response:
            payload = json.loads(response.read().decode("utf-8-sig"))
        if payload.get("BEAAPI", {}).get("Error"):
            continue
        data = payload["BEAAPI"]["Results"].get("Data", [])
        rows.extend([data] if isinstance(data, dict) else data)
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    df["line"] = pd.to_numeric(df["LineNumber"], errors="coerce")
    df["date"] = pd.to_datetime(df["TimePeriod"].astype(str).str.replace(
        r"^(\d{4})M(\d{1,2})$", r"\1-\2-01", regex=True))
    df["value"] = pd.to_numeric(
        df["DataValue"].astype(str).str.replace(",", ""), errors="coerce")
    matrix = (df.dropna(subset=["line"])
              .pivot_table(index="date", columns="line", values="value",
                           aggfunc="first").sort_index())
    matrix.columns = matrix.columns.astype(int)
    matrices[label] = matrix
    print(f"{table_name} ({label}): {matrix.shape[0]} months x {matrix.shape[1]} lines")

## 4. Construct the finest usable signed partition


In [ ]:
STOP_AT = {13, 17, 342}
LAST12 = matrices["nominal"].index[-12:]


def snom(L):
    """Signed nominal spending, averaged over the last 12 months."""
    return sign[L] * matrices["nominal"][L].reindex(LAST12).mean()


def usable(L):
    if L not in matrices["prices"].columns or L not in matrices["nominal"].columns:
        return False
    return (matrices["prices"][L].notna().mean() > 0.99
            and matrices["nominal"][L].notna().mean() > 0.99)


def expand(L):
    kids = children.get(L, [])
    if not kids or L in STOP_AT:
        return [L] if usable(L) else []
    if all(usable(k) or children.get(k) for k in kids):
        out = []
        for k in kids:
            e = expand(k)
            if not e:
                break
            out.extend(e)
        else:
            if abs(sum(snom(c) for c in out) - snom(L)) < max(2.0, 0.002 * abs(snom(L))):
                return out
    return [L] if usable(L) else []


partition = expand(1)
core_excl = descendants(73) | {73} | descendants(113) | {113} | {170, 171}
core_lines = sorted(set(partition) - core_excl)

official_core = snom(1) - snom(73) - snom(113) - snom(170) - snom(171)
core_cov = sum(snom(L) for L in core_lines)
print(f"partition: {len(partition)} components, core: {len(core_lines)}")
print(f"core coverage: {core_cov:,.0f} of official {official_core:,.0f} "
      f"({100 * core_cov / official_core:.2f}%)")
neg = [L for L in core_lines if snom(L) < 0]
print(f"negative-weight components: {len(neg)} (nonresident spending, insurance "
      f"losses, remittances, employee reimbursement)")

## 5. CPI/PPI crosswalk

The embedded crosswalk makes this notebook independent of the other notebooks.


In [ ]:
CROSSWALK = {
    # --- Motor vehicles and parts ---
    7:   ("CPI", {"CUSR0000SETA01": 1.0}),   # New domestic autos -> new vehicles
    8:   ("CPI", {"CUSR0000SETA01": 1.0}),   # New foreign autos -> new vehicles
    9:   ("CPI", {"CUSR0000SS45021": 1.0}),  # New light trucks -> new trucks
    13:  ("CPI", {"CUSR0000SETA02": 1.0}),   # Used autos -> used cars and trucks
    17:  ("CPI", {"CUSR0000SETA02": 1.0}),   # Used light trucks
    21:  ("CPI", {"CUSR0000SETC01": 1.0}),   # Tires
    22:  ("CPI", {"CUUR0000SETC02": 1.0}),   # Accessories and parts
    # --- Furnishings and durable household equipment ---
    25:  ("CPI", {"CUSR0000SEHJ": 1.0}),     # Furniture -> furniture and bedding
    26:  ("CPI", {"CUUR0000SEHL01": 1.0}),   # Clocks, lamps, decorator items
    27:  ("CPI", {"CUUR0000SEHH01": 1.0}),   # Carpets -> floor coverings
    28:  ("CPI", {"CUSR0000SEHH02": 1.0}),   # Window coverings
    30:  ("CPI", {"CUSR0000SEHK01": 1.0}),   # Major household appliances
    31:  ("CPI", {"CUSR0000SEHK02": 1.0}),   # Small electric appliances -> other appliances
    33:  ("NONE", {}),                       # Dishes and flatware (fallback fits better)
    34:  ("NONE", {}),                       # Nonelectric cookware (fallback fits better)
    36:  ("CPI", {"CUSR0000SEHM01": 1.0}),   # Tools, hardware and supplies
    37:  ("CPI", {"CUSR0000SEHM02": 1.0}),   # Outdoor equipment and supplies
    # --- Recreational goods and vehicles ---
    41:  ("CPI", {"CUSR0000SERA01": 1.0}),   # Televisions
    42:  ("CPI", {"CUSR0000SERA03": 1.0}),   # Other video equipment
    43:  ("CPI", {"CUSR0000SERA05": 1.0}),   # Audio equipment
    45:  ("CPI", {"CUUR0000SERA06": 1.0}),   # Audio media -> recorded music
    46:  ("CPI", {"CUUR0000SERA04": 1.0}),   # Video media
    47:  ("CPI", {"CUSR0000SERD01": 1.0}),   # Photographic equipment
    49:  ("CPI", {"CUSR0000SEEE01": 1.0}),   # Personal computers and peripherals
    50:  ("CPI", {"CUUR0000SEEE02": 1.0}),   # Computer software and accessories
    51:  ("CPI", {"CUSR0000SEEE04": 1.0}),   # Calculators/other info processing
    52:  ("CPI", {"CUSR0000SERC02": 1.0}),   # Sporting equipment -> sports equipment
    54:  ("CPI", {"CUSR0000SERC01": 1.0}),   # Motorcycles -> sports vehicles incl bicycles
    55:  ("CPI", {"CUSR0000SERC01": 1.0}),   # Bicycles
    57:  ("CPI", {"CUSR0000SERC01": 1.0}),   # Pleasure boats
    58:  ("CPI", {"CUSR0000SERC01": 1.0}),   # Pleasure aircraft
    59:  ("CPI", {"CUSR0000SERC01": 1.0}),   # Other recreational vehicles
    60:  ("CPI", {"CUUR0000SERG02": 1.0}),   # Recreational books
    61:  ("CPI", {"CUSR0000SERE03": 1.0}),   # Musical instruments
    64:  ("CPI", {"CUSR0000SEAG02": 1.0}),   # Jewelry
    65:  ("CPI", {"CUSR0000SEAG01": 1.0}),   # Watches
    # --- Other durable goods ---
    67:  ("CPI", {"CUUR0000SEMG": 1.0}),     # Therapeutic medical equipment
    68:  ("CPI", {"CUSR0000SEMC04": 1.0}),   # Eyeglasses and eye care
    69:  ("CPI", {"CUSR0000SEEA": 1.0}),     # Educational books and supplies
    70:  ("CPI", {"CUSR0000SEGE": 1.0}),     # Luggage -> misc personal goods
    71:  ("CPI", {"CUSR0000SEEE04": 1.0}),   # Telephone equipment
    # --- Food at home ---
    77:  ("CPI", {"CUSR0000SEFA": 1.0}),     # Cereals
    78:  ("CPI", {"CUSR0000SEFB": 1.0}),     # Bakery products
    80:  ("CPI", {"CUSR0000SEFC": 1.0}),     # Beef and veal
    81:  ("CPI", {"CUSR0000SEFD": 1.0}),     # Pork
    82:  ("CPI", {"CUSR0000SEFE": 1.0}),     # Other meats
    83:  ("CPI", {"CUSR0000SEFF": 1.0}),     # Poultry
    84:  ("CPI", {"CUSR0000SEFG": 1.0}),     # Fish and seafood
    86:  ("CPI", {"CUSR0000SEFJ01": 1.0}),   # Fresh milk -> milk
    87:  ("CPI", {"CUSR0000SEFJ": 1.0}),     # Processed dairy -> dairy products
    88:  ("CPI", {"CUSR0000SEFH": 1.0}),     # Eggs
    89:  ("CPI", {"CUSR0000SEFS": 1.0}),     # Fats and oils
    91:  ("CPI", {"CUSR0000SEFK": 1.0}),     # Fresh fruit
    92:  ("CPI", {"CUSR0000SEFL": 1.0}),     # Fresh vegetables
    93:  ("CPI", {"CUSR0000SEFM": 1.0}),     # Processed fruits and vegetables
    94:  ("CPI", {"CUSR0000SEFR": 1.0}),     # Sugar and sweets
    95:  ("CPI", {"CUSR0000SEFT": 1.0}),     # Food nec -> other foods
    97:  ("CPI", {"CUSR0000SEFP": 1.0}),     # Coffee, tea -> beverage materials
    98:  ("CPI", {"CUSR0000SEFN": 1.0}),     # Soft drinks -> nonalcoholic beverages
    100: ("CPI", {"CUSR0000SEFW02": 1.0}),   # Spirits at home
    101: ("CPI", {"CUSR0000SEFW03": 1.0}),   # Wine at home
    102: ("CPI", {"CUSR0000SEFW01": 1.0}),   # Beer at home
    103: ("NONE", {}),                       # Food produced/consumed on farms
    # --- Clothing and footwear ---
    106: ("CPI", {"CUSR0000SEAC": 1.0}),     # Women's and girls'
    107: ("CPI", {"CUSR0000SEAA": 1.0}),     # Men's and boys'
    108: ("CPI", {"CUSR0000SEAF": 1.0}),     # Children's and infants'
    110: ("NONE", {}),                       # Clothing materials (no live CPI series)
    111: ("NONE", {}),                       # Military clothing
    112: ("CPI", {"CUSR0000SEAE": 1.0}),     # Footwear
    # --- Energy goods ---
    115: ("CPI", {"CUSR0000SETB01": 1.0}),   # Gasoline
    116: ("CPI", {"CUUR0000SS47021": 1.0}),  # Lubricants -> motor oil, coolant
    118: ("CPI", {"CUSR0000SEHE01": 1.0}),   # Fuel oil
    119: ("CPI", {"CUSR0000SEHE02": 1.0}),   # Other fuels -> propane, kerosene, firewood
    # --- Medical goods ---
    123: ("CPI", {"CUSR0000SEMF01": 1.0}),   # Prescription drugs
    124: ("CPI", {"CUSR0000SEMF02": 1.0}),   # Nonprescription drugs
    125: ("CPI", {"CUUR0000SEMG": 1.0}),     # Other medical products
    # --- Other nondurable goods ---
    127: ("CPI", {"CUSR0000SERE01": 1.0}),   # Games, toys -> toys
    128: ("CPI", {"CUSR0000SERB01": 1.0}),   # Pets and related products
    129: ("CPI", {"CUSR0000SEHL02": 1.0}),   # Flowers, seeds, potted plants
    130: ("CPI", {"CUUR0000SERD02": 1.0}),   # Film and photographic supplies
    132: ("CPI", {"CUSR0000SEHN01": 1.0}),   # Household cleaning products
    133: ("CPI", {"CUUR0000SEHN02": 1.0}),   # Household paper products
    134: ("CPI", {"CUSR0000SEHH03": 1.0}),   # Household linens -> other linens
    135: ("NONE", {}),                       # Sewing items (no live CPI series)
    136: ("CPI", {"CUSR0000SEHN03": 1.0}),   # Misc household products
    138: ("CPI", {"CUUR0000SEGB01": 1.0}),   # Hair/dental/shaving products
    139: ("CPI", {"CUUR0000SEGB02": 1.0}),   # Cosmetics
    140: ("NONE", {}),                       # Electric personal care (fallback fits better)
    141: ("CPI", {"CUSR0000SEGA": 1.0}),     # Tobacco
    143: ("CPI", {"CUUR0000SERG01": 1.0}),   # Newspapers and periodicals
    144: ("CPI", {"CUSR0000SEGE": 1.0}),     # Stationery -> misc personal goods
    # --- Housing ---
    155: ("CPI", {"CUSR0000SEHA": 1.0}),     # Tenant mobile homes -> rent
    157: ("CPI", {"CUSR0000SEHA": 1.0}),     # Tenant stationary homes -> rent
    158: ("CPI", {"CUSR0000SEHA": 1.0}),     # Tenant landlord durables -> rent
    161: ("CPI", {"CUSR0000SEHC": 1.0}),     # Owner mobile homes -> OER
    162: ("CPI", {"CUSR0000SEHC": 1.0}),     # Owner stationary homes -> OER
    163: ("NONE", {}),                       # Farm dwellings
    164: ("NONE", {}),                       # Group housing
    # --- Household utilities ---
    167: ("CPI", {"CUSR0000SEHG01": 1.0}),   # Water and sewer
    168: ("CPI", {"CUSR0000SEHG02": 1.0}),   # Garbage and trash
    170: ("CPI", {"CUSR0000SEHF01": 1.0}),   # Electricity
    171: ("CPI", {"CUSR0000SEHF02": 1.0}),   # Natural gas
    # --- Health care services (PPI per BEA methodology) ---
    174: ("PPI", {"PCU621111621111": 1.0}),  # Physician services
    175: ("CPI", {"CUSR0000SEMC02": 1.0}),   # Dental services (CPI fits BEA best)
    176: ("PPI", {"PCU621610621610": 0.5,    # Paramedical: home health
                  "PCU621511621511": 0.5}),  #   + medical labs
    184: ("PPI", {"PCU622110622110": 1.0}),  # Nonprofit hospitals
    185: ("PPI", {"PCU622110622110": 1.0}),  # Proprietary hospitals
    186: ("PPI", {"PCU622110622110": 1.0}),  # Government hospitals
    187: ("PPI", {"PCU623110623110": 1.0}),  # Nursing homes
    # --- Transportation services ---
    192: ("CPI", {"CUSR0000SETD": 1.0}),     # Motor vehicle maintenance/repair
    194: ("CPI", {"CUSR0000SETA03": 1.0}),   # Motor vehicle leasing -> leased cars
    197: ("CPI", {"CUSR0000SETA04": 1.0}),   # Motor vehicle rental -> car rental
    198: ("CPI", {"CUSR0000SS52051": 1.0}),  # Parking fees and tolls
    201: ("CPI", {"CUSR0000SETG02": 1.0}),   # Railway -> other intercity transport
    203: ("CPI", {"CUSR0000SETG02": 1.0}),   # Intercity buses
    204: ("NONE", {}),                       # Taxis (CPI series ended 2019)
    205: ("CPI", {"CUUR0000SETG03": 1.0}),   # Intracity mass transit
    206: ("NONE", {}),                       # Other road transportation
    207: ("PPI", {"PCU481111481111": 1.0}),  # Air transportation (PPI per BEA)
    208: ("CPI", {"CUSR0000SS53023": 1.0}),  # Water transportation -> ship fare
    # --- Recreation services ---
    211: ("CPI", {"CUSR0000SERF01": 1.0}),   # Membership clubs, sports centers
    212: ("CPI", {"CUSR0000SERF02": 1.0}),   # Amusement parks -> admissions
    214: ("CPI", {"CUSR0000SS62031": 1.0}),  # Movie theaters
    215: ("CPI", {"CUSR0000SS62031": 1.0}),  # Live entertainment
    216: ("NONE", {}),                       # Spectator sports (fallback fits better)
    217: ("NONE", {}),                       # Museums and libraries
    218: ("CPI", {"CUSR0000SERA02": 1.0}),   # AV services -> cable/satellite/streaming
    227: ("NONE", {}),                       # Casino gambling
    228: ("NONE", {}),                       # Lotteries
    229: ("NONE", {}),                       # Pari-mutuel
    231: ("CPI", {"CUSR0000SERB02": 1.0}),   # Veterinary and pet services
    232: ("NONE", {}),                       # Package tours
    233: ("NONE", {}),                       # Repair of rec vehicles/equipment
    # --- Food services and accommodations ---
    239: ("NONE", {}),                       # Elem/secondary school lunches (no CPI fit)
    240: ("NONE", {}),                       # Higher ed school lunches (no CPI fit)
    241: ("CPI", {"CUSR0000SEFV": 1.0}),     # Other purchased meals -> food away
    245: ("CPI", {"CUSR0000SEFX": 1.0}),     # Alcohol in purchased meals
    247: ("NONE", {}),                       # Food supplied to civilians
    248: ("NONE", {}),                       # Food supplied to military
    250: ("CPI", {"CUSR0000SEHB02": 1.0}),   # Hotels and motels
    251: ("CPI", {"CUSR0000SEHB01": 1.0}),   # Housing at schools
    # --- Financial services and insurance ---
    255: ("NONE", {}),                       # Commercial banks (imputed)
    256: ("NONE", {}),                       # Other depository (imputed)
    257: ("NONE", {}),                       # Pension funds (imputed)
    258: ("PPI", {"PCU5239405239401": 0.6,   # Financial charges: portfolio mgmt
                  "WPU40110101": 0.4}),      #   + brokerage/investment advice PPI
    271: ("NONE", {}),                       # Life insurance (imputed)
    272: ("PPI", {"PCU9241269241262": 1.0}), # Household insurance -> homeowners PPI
    275: ("PPI", {"PCU524114524114": 1.0}),  # Health insurance -> direct health PPI
    279: ("PPI", {"PCU9241269241261": 1.0}), # Motor vehicle insurance -> auto PPI
    # --- Communication ---
    281: ("CPI", {"CUUR0000SEED": 0.6,       # Telephone services
                  "CUSR0000SEEE03": 0.3,     # Internet services
                  "CUSR0000SEEC01": 0.1}),   # Postage
    # --- Education ---
    292: ("CPI", {"CUSR0000SEEB01": 1.0}),   # Public/proprietary higher ed -> college tuition
    293: ("CPI", {"CUSR0000SEEB01": 1.0}),   # Nonprofit higher ed
    295: ("CPI", {"CUSR0000SEEB02": 1.0}),   # Elementary and secondary schools
    296: ("CPI", {"CUSR0000SEEB03": 1.0}),   # Day care and nursery -> day care/preschool
    297: ("CPI", {"CUSR0000SEEB04": 1.0}),   # Commercial/vocational schools
    # --- Professional and personal services ---
    299: ("CPI", {"CUSR0000SEGD01": 1.0}),   # Legal services (ends 2025 -> fallback)
    301: ("NONE", {}),                       # Tax preparation (fallback fits better)
    302: ("NONE", {}),                       # Employment agency services
    303: ("NONE", {}),                       # Other personal business services
    304: ("NONE", {}),                       # Labor organization dues
    305: ("NONE", {}),                       # Professional association dues
    306: ("CPI", {"CUSR0000SEGD02": 1.0}),   # Funeral and burial
    309: ("CPI", {"CUUR0000SEGC01": 1.0}),   # Hairdressing -> haircuts
    310: ("CPI", {"CUUR0000SEGC": 1.0}),     # Misc personal care services
    312: ("CPI", {"CUSR0000SEGD03": 1.0}),   # Laundry and dry cleaning
    313: ("CPI", {"CUUR0000SEGD04": 1.0}),   # Clothing repair/rental
    314: ("CPI", {"CUUR0000SEGD04": 1.0}),   # Footwear repair
    316: ("CPI", {"CUSR0000SEEB03": 1.0}),   # Child care -> day care/preschool
    317: ("NONE", {}),                       # Social assistance
    324: ("NONE", {}),                       # Social advocacy orgs
    325: ("NONE", {}),                       # Religious organizations
    326: ("NONE", {}),                       # Foundations
    # --- Household services ---
    328: ("CPI", {"CUUR0000SEHP01": 1.0}),   # Domestic services (ends 2025 -> fallback)
    329: ("CPI", {"CUSR0000SEHP03": 1.0}),   # Moving, storage, freight
    330: ("CPI", {"CUUR0000SEHP04": 1.0}),   # Repair of furniture (ends 2025)
    331: ("CPI", {"CUUR0000SEHP04": 1.0}),   # Repair of appliances (ends 2025)
    332: ("NONE", {}),                       # Other household services
    # --- NPISH ---
    342: ("NONE", {}),                       # NPISH final consumption (imputed)
}


OVERRIDES = {
    # BLS-published seasonally adjusted commodity PPIs (WPS...) replace the NSA
    # industry PPIs where they exist - BLS's own factors beat the crude adjustment.
    174: ("PPI", {"WPS511101": 1.0}),         # physician care, SA
    184: ("PPI", {"WPS512101": 1.0}),         # hospital inpatient care, SA
    185: ("PPI", {"WPS512101": 1.0}),
    186: ("PPI", {"WPS512101": 1.0}),
    187: ("PPI", {"WPS512102": 1.0}),         # nursing home care, SA
    188: ("PPI", {"WPS512102": 1.0}),
    189: ("PPI", {"WPS512102": 1.0}),
    207: ("PPI", {"WPS302201": 1.0}),         # airline passenger services, SA
    177: ("PPI", {"PCU621610621610": 1.0}),   # home health care
    178: ("PPI", {"PCU621511621511": 1.0}),   # medical laboratories
    220: ("NONE", {}),                        # photo processing
    221: ("NONE", {}),                        # photo studios
    222: ("NONE", {}),                        # repair/rental of AV equipment
    225: ("CPI", {"CUUR0000SERA06": 1.0}),    # audio streaming -> recorded music/subscr
    242: ("CPI", {"CUUR0000SEFV02": 1.0}),    # limited service meals
    243: ("CPI", {"CUSR0000SEFV01": 1.0}),    # meals at other eating places -> full service
    259: ("NONE", {}),                        # bank service charges (checking CPI fits poorly)
    262: ("PPI", {"WPU40110101": 1.0}),       # exchange-listed equities -> brokerage
    263: ("PPI", {"WPU40110101": 1.0}),       # other direct commissions
    265: ("PPI", {"WPU40110101": 1.0}),       # OTC equities (imputed commissions)
    266: ("PPI", {"WPU40110101": 1.0}),       # other imputed commissions
    267: ("PPI", {"WPU40110101": 1.0}),       # mutual fund sales charges
    268: ("PPI", {"PCU5239405239401": 1.0}),  # portfolio management and inv advice
    269: ("PPI", {"PCU5239405239401": 1.0}),  # trust, fiduciary, custody
    273: ("PPI", {"PCU9241269241262": 1.0}),  # household insurance premiums
    274: ("NONE", {}),                        # less: household insurance normal losses
    276: ("PPI", {"PCU524114524114": 1.0}),   # health ins: medical & hospitalization
    277: ("NONE", {}),                        # health ins: income loss
    278: ("NONE", {}),                        # workers' compensation
    283: ("CPI", {"CUUR0000SEED": 1.0}),      # landline local (no separate CPI)
    284: ("CPI", {"CUUR0000SEED": 1.0}),      # landline long-distance
    285: ("CPI", {"CUUR0000SEED03": 1.0}),    # cellular -> wireless CPI
    287: ("CPI", {"CUSR0000SEEC01": 1.0}),    # first-class postal -> postage
    288: ("CPI", {"CUUR0000SEEC02": 1.0}),    # other delivery services
    289: ("CPI", {"CUSR0000SEEE03": 1.0}),    # internet access
    335: ("NONE", {}),                        # foreign passenger fares (air PPI fits poorly)
    336: ("NONE", {}), 337: ("NONE", {}),     # travel/study abroad
    339: ("NONE", {}), 340: ("NONE", {}), 341: ("NONE", {}),  # nonresident spending
    147: ("NONE", {}), 148: ("NONE", {}), 149: ("NONE", {}),  # net expenditures abroad
    156: ("CPI", {"CUSR0000SEHA": 1.0}),      # tenant stationary homes -> rent
}

XWALK = {}
for L in partition:
    if L in OVERRIDES:
        XWALK[L] = OVERRIDES[L]
    elif L in CROSSWALK:
        XWALK[L] = CROSSWALK[L]
    else:
        a = next((a for a in ancestors(L) if a in CROSSWALK), None)
        if a is not None:
            XWALK[L] = CROSSWALK[a]        # inherit the parent component's source
        else:
            d = sorted(descendants(L) & set(CROSSWALK))
            assert d, f"no mapping found for line {L}"
            XWALK[L] = CROSSWALK[d[0]]     # e.g. 156 <- 157 (rent)

n_none = sum(1 for L in core_lines if XWALK[L][0] == "NONE")
print(f"{len(core_lines)} core components: {len(core_lines) - n_none} mapped to "
      f"CPI/PPI, {n_none} on fallback")

## 6. Pull source indexes and audit their coverage


In [ ]:
all_series = sorted(
    {sid for L in core_lines for sid in XWALK[L][1]} | {HOSPITAL_ALT_SERIES}
)
source_levels, missing_series = bls_fetch(
    all_series, SOURCE_START_YEAR, date.today().year
)

for sid in list(missing_series):
    try:
        s = fred_fetch(sid)
        if len(s):
            source_levels[sid] = s
            missing_series.remove(sid)
            print(f"{sid}: recovered from FRED")
    except Exception as exc:
        print(f"{sid}: FRED recovery failed ({type(exc).__name__})")
source_levels = source_levels.sort_index()

print(
    f"pulled {source_levels.shape[1]} of {len(all_series)} series, "
    f"{source_levels.index[0]:%Y-%m} .. {source_levels.index[-1]:%Y-%m}"
)
if missing_series:
    print("STILL MISSING (will use fallback):", missing_series)

report = pd.DataFrame({
    "first": source_levels.apply(lambda s: s.first_valid_index()),
    "last": source_levels.apply(lambda s: s.last_valid_index()),
})
stale = report[report["last"] < source_levels.index[-1] - pd.DateOffset(months=2)]
if len(stale):
    print("\nSeries no longer publishing or materially stale:")
    print(stale.to_string())


## 7. Construct causal monthly source changes


In [ ]:
def causal_own_history_sa(raw_changes, window_years=5):
    """Causal calendar-month adjustment: every value uses observations through t-1."""
    seasonal = raw_changes.groupby(raw_changes.index.month).transform(
        lambda x: x.shift(1).rolling(window_years, min_periods=3).mean()
    )
    average = raw_changes.shift(1).rolling(60, min_periods=36).mean()
    adjusted = raw_changes - seasonal + average
    return adjusted.where(adjusted.notna(), raw_changes)


source_chg = source_levels.pct_change(fill_method=None)

# Detailed NSA CPI items benefit from a causal own-history seasonal adjustment.
nsa_cpi = [c for c in source_chg.columns if c.startswith("CUUR")]
if nsa_cpi:
    source_chg[nsa_cpi] = causal_own_history_sa(source_chg[nsa_cpi])

# Hospital services are the important exception among industry PPIs: the causal
# adjustment materially improves line-level fit and removes the WPS series' bias.
if HOSPITAL_ALT_SERIES in source_chg:
    source_chg[[HOSPITAL_ALT_SERIES]] = causal_own_history_sa(
        source_chg[[HOSPITAL_ALT_SERIES]]
    )

print(f"causal adjustment applied to {len(nsa_cpi)} NSA CPI series")
print(f"hospital alternative prepared from {HOSPITAL_ALT_SERIES}")


## 8. Actual component changes and signed Fisher-approximation weights


In [ ]:
sgn = pd.Series({L: sign[L] for L in partition})
prices = matrices["prices"][partition].copy()
nominal = matrices["nominal"][partition].mul(sgn, axis=1)
real = matrices["real"].reindex(columns=partition).mul(sgn, axis=1)

quantity = real.combine_first(nominal / prices)
common = prices.index.intersection(quantity.index)
prices = prices.loc[common].sort_index()
quantity = quantity.loc[common].sort_index()

price_changes = prices.pct_change().iloc[1:]

p_lag = prices.shift(1).iloc[1:].to_numpy(dtype=float)
q_cur = quantity.iloc[1:].to_numpy(dtype=float)
q_lag = quantity.shift(1).iloc[1:].to_numpy(dtype=float)
w1 = q_cur * p_lag
w1 = w1 / w1.sum(axis=1, keepdims=True)
w2 = q_lag * p_lag
w2 = w2 / w2.sum(axis=1, keepdims=True)
weights = pd.DataFrame(0.5 * w1 + 0.5 * w2,
                       index=price_changes.index, columns=prices.columns)

core_weights = weights[core_lines]
core_weights = core_weights.div(core_weights.sum(axis=1), axis=0)

last_bea = price_changes.index[-1]
last_source = source_chg.index[-1]
print(f"BEA actuals through {last_bea:%Y-%m}; CPI/PPI sources through {last_source:%Y-%m}")

## 9. Raw proxy forecasts, fallback, and hospital alternative


In [ ]:
nowcast_index = pd.date_range(
    BACKTEST_START, max(last_source, last_bea), freq="MS"
)

# Preserve missing proxies here. They are filled only after the alternative and
# adaptive forecasts have been constructed, so a failed release is never mistaken
# for a genuine zero or source signal.
proxy_wps = pd.DataFrame(index=nowcast_index, columns=core_lines, dtype=float)
for line in core_lines:
    src_kind, ids = XWALK[line]
    if src_kind == "NONE":
        continue
    total = sum(ids.values())
    proxy_wps[line] = sum(
        source_chg[sid].reindex(nowcast_index) * (weight / total)
        for sid, weight in ids.items()
    )

# Competing hospital specification. Everything else is identical.
proxy_pcu = proxy_wps.copy()
for line in [184, 185, 186]:
    if line in proxy_pcu:
        proxy_pcu[line] = source_chg[HOSPITAL_ALT_SERIES].reindex(nowcast_index)

pc_ext = price_changes.reindex(price_changes.index.union(nowcast_index))
fallback = (
    pc_ext.rolling(FALLBACK_WINDOW, min_periods=3)
    .mean().shift(1).reindex(nowcast_index)
)

raw_from_fallback = proxy_wps.isna()
raw_nowcast_chg = proxy_wps.where(proxy_wps.notna(), fallback[core_lines])

latest_weights = core_weights.reindex(nowcast_index).ffill()
fallback_abs_weight = (latest_weights.abs() * raw_from_fallback).sum(axis=1)
fallback_signed_weight = (latest_weights * raw_from_fallback).sum(axis=1)
print(f"raw cells on fallback: {100 * raw_from_fallback.mean().mean():.1f}%")
print(
    "latest fallback weight: "
    f"{100 * fallback_signed_weight.iloc[-1]:.1f}% signed, "
    f"{100 * fallback_abs_weight.iloc[-1]:.1f}% absolute"
)
assert not raw_nowcast_chg.isna().any().any(), "unfilled raw nowcast cells remain"


## 10. Core aggregation helper


In [ ]:
def core_aggregate(changes, wts):
    """Weighted mean of one month's core component price changes."""
    w = wts.reindex(changes.index).to_numpy(dtype=float)
    return float((changes.to_numpy(dtype=float) * w).sum() / w.sum())

## 11. Adaptive component shrinkage


In [ ]:
def adaptive_component_forecast(proxy, fallback_values, actual_values):
    """Causally blend each proxy with its BEA trend using component errors through t-1.

    alpha=1 is the CPI/PPI proxy; alpha=0 is the six-month BEA trend. The clipped
    least-squares weight is estimated separately by component over at most 120
    prior months. Components with fewer than 24 observations remain unshrunk.
    """
    proxy = proxy.reindex(index=nowcast_index, columns=core_lines)
    fb = fallback_values.reindex(index=nowcast_index, columns=core_lines)
    actual = actual_values.reindex(index=nowcast_index, columns=core_lines)

    delta = proxy - fb
    target = actual - fb
    numerator = (delta * target).shift(1).rolling(
        ADAPT_WINDOW, min_periods=ADAPT_MIN_OBS
    ).sum()
    denominator = (delta * delta).shift(1).rolling(
        ADAPT_WINDOW, min_periods=ADAPT_MIN_OBS
    ).sum()
    alpha = (numerator / denominator).clip(0.0, 1.0).fillna(1.0)
    alpha = alpha.where(proxy.notna(), 0.0)

    estimate = alpha * proxy + (1.0 - alpha) * fb
    estimate = estimate.where(estimate.notna(), fb)
    return estimate, alpha


adaptive_wps_chg, alpha_wps = adaptive_component_forecast(
    proxy_wps, fallback, price_changes
)
adaptive_pcu_chg, alpha_pcu = adaptive_component_forecast(
    proxy_pcu, fallback, price_changes
)

assert not adaptive_wps_chg.isna().any().any()
assert not adaptive_pcu_chg.isna().any().any()

mapped_cols = [c for c in core_lines if XWALK[c][0] != "NONE"]
latest_alpha = alpha_wps.loc[alpha_wps.index[-1], mapped_cols]
print(
    "latest mapped-component proxy weight: "
    f"mean={latest_alpha.mean():.3f}, median={latest_alpha.median():.3f}, "
    f"min={latest_alpha.min():.3f}"
)


## 12. Online hospital ensemble, bias correction, and backtest


In [ ]:
def aggregate_with_available_weights(changes):
    out = {}
    for t in changes.index:
        t_prev = t - pd.DateOffset(months=1)
        if t_prev in core_weights.index:
            out[t] = core_aggregate(changes.loc[t], core_weights.loc[t_prev])
        elif t > last_bea:
            out[t] = core_aggregate(changes.loc[t], core_weights.loc[last_bea])
    return pd.Series(out, dtype=float)


core_index = fred_fetch("PCEPILFE", start="2013-01-01")
actual_mm = core_index.pct_change()

raw_mm = aggregate_with_available_weights(raw_nowcast_chg)
adaptive_wps_mm = aggregate_with_available_weights(adaptive_wps_chg)
adaptive_pcu_mm = aggregate_with_available_weights(adaptive_pcu_chg)

# Online hospital-model ensemble. At month t the weight uses aggregate errors only
# through t-1. The two candidates differ only in hospital deflators.
hosp_delta = adaptive_wps_mm - adaptive_pcu_mm
hosp_target = actual_mm.reindex(hosp_delta.index) - adaptive_pcu_mm
hosp_num = (hosp_delta * hosp_target).shift(1).rolling(
    HOSPITAL_ENSEMBLE_WINDOW, min_periods=ADAPT_MIN_OBS
).sum()
hosp_den = (hosp_delta * hosp_delta).shift(1).rolling(
    HOSPITAL_ENSEMBLE_WINDOW, min_periods=ADAPT_MIN_OBS
).sum()
hospital_wps_weight = (hosp_num / hosp_den).clip(0.0, 1.0).fillna(0.5)
ensemble_mm = adaptive_pcu_mm + hospital_wps_weight * hosp_delta

# Small rolling bias correction, again based only on residuals known before t.
online_bias = (ensemble_mm - actual_mm.reindex(ensemble_mm.index)).shift(1).rolling(
    BIAS_WINDOW, min_periods=BIAS_MIN_OBS
).mean()
final_mm = ensemble_mm - online_bias.fillna(0.0)

# A component representation of the ensemble (before the aggregate bias correction).
component_ensemble = adaptive_pcu_chg.add(
    adaptive_wps_chg.sub(adaptive_pcu_chg).mul(hospital_wps_weight, axis=0)
)
component_ensemble_mm = aggregate_with_available_weights(component_ensemble)
assert np.nanmax(np.abs(component_ensemble_mm - ensemble_mm)) < 1e-10

# Same-month-weight component replication remains a useful aggregation diagnostic.
replication = pd.Series({
    t: core_aggregate(price_changes.loc[t, core_lines], core_weights.loc[t])
    for t in price_changes.index.intersection(nowcast_index)
})

bt = pd.DataFrame({
    "raw_mm": raw_mm,
    "adaptive_wps_mm": adaptive_wps_mm,
    "adaptive_pcu_mm": adaptive_pcu_mm,
    "ensemble_mm": ensemble_mm,
    "online_bias": online_bias,
    "final_mm": final_mm,
    "replication_mm": replication,
    "actual_mm": actual_mm,
})

for col in ["raw_mm", "adaptive_wps_mm", "adaptive_pcu_mm", "ensemble_mm",
            "final_mm", "replication_mm", "actual_mm"]:
    bt[col.replace("_mm", "_ann")] = 100 * ((1 + bt[col]) ** 12 - 1)
bt["naive_lag_ann"] = bt["actual_ann"].shift(1)
bt["naive_6m_ann"] = bt["actual_ann"].shift(1).rolling(6).mean()


def score_table(frame, start):
    rows = {}
    actual = frame.loc[start:, "actual_mm"]
    for col in ["raw_mm", "adaptive_wps_mm", "adaptive_pcu_mm", "ensemble_mm",
                "final_mm", "replication_mm"]:
        pair = pd.concat([frame.loc[start:, col], actual], axis=1).dropna()
        error = pair.iloc[:, 0] - pair.iloc[:, 1]
        ann_error = (
            100 * ((1 + pair.iloc[:, 0]) ** 12 - 1)
            - 100 * ((1 + pair.iloc[:, 1]) ** 12 - 1)
        )
        rows[col] = {
            "months": len(pair),
            "MAE_mom_bp": 10000 * error.abs().mean(),
            "RMSE_mom_bp": 10000 * np.sqrt((error ** 2).mean()),
            "bias_mom_bp": 10000 * error.mean(),
            "MAE_ann_pp": ann_error.abs().mean(),
            "corr": pair.iloc[:, 0].corr(pair.iloc[:, 1]),
        }
    return pd.DataFrame(rows).T


metrics = score_table(bt, EVALUATION_START)
print(f"Pseudo-out-of-sample-style scoring, {EVALUATION_START:%Y-%m} onward")
display(metrics.round(3))

improvement = 100 * (
    1 - metrics.loc["final_mm", "MAE_mom_bp"] / metrics.loc["raw_mm", "MAE_mom_bp"]
)
print(f"Final versus raw monthly MAE improvement: {improvement:.1f}%")


## 13. Visual comparison


In [ ]:
scored = bt.loc[EVALUATION_START:].dropna(subset=["actual_mm", "final_mm"])
rolling_raw = 10000 * (scored["raw_mm"] - scored["actual_mm"]).abs().rolling(24).mean()
rolling_final = 10000 * (scored["final_mm"] - scored["actual_mm"]).abs().rolling(24).mean()

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), width_ratios=[2, 1])
    axes[0].plot(scored.index, scored["actual_ann"], label="Actual", lw=1.5)
    axes[0].plot(scored.index, scored["raw_ann"], label="Raw component model", alpha=0.65)
    axes[0].plot(scored.index, scored["final_ann"], label="Improved model", lw=1.25)
    axes[0].set_title("Core PCE: 1-month annualized")
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    axes[1].plot(rolling_raw, label="Raw")
    axes[1].plot(rolling_final, label="Improved")
    axes[1].set_title("24-month rolling MAE (m/m bp)")
    axes[1].grid(alpha=0.25)
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib is not installed; calculations completed and chart was skipped")


## 14. Component contribution diagnostics


In [ ]:
window = component_ensemble.index.intersection(price_changes.index)
component_error = (
    component_ensemble.loc[window, core_lines]
    - price_changes.loc[window, core_lines]
)
prior_w = core_weights.shift(1).reindex(window)
contribution_error = component_error * prior_w
aggregate_component_error = contribution_error.sum(axis=1)

diagnostics = pd.DataFrame({
    "source": {c: XWALK[c][0] for c in core_lines},
    "mean_abs_weight": prior_w.abs().mean(),
    "component_rmse_pct": 100 * np.sqrt((component_error ** 2).mean()),
    "component_corr": {
        c: component_ensemble.loc[window, c].corr(price_changes.loc[window, c])
        for c in core_lines
    },
    "mean_proxy_weight": alpha_wps.loc[window].mean(),
    # E[c_i * total error] sums to the component-ensemble aggregate MSE.
    "aggregate_mse_contribution": {
        c: (contribution_error[c] * aggregate_component_error).mean()
        for c in core_lines
    },
})
diagnostics["description"] = descriptions.reindex(core_lines).str.slice(0, 55)
aggregate_mse = (aggregate_component_error ** 2).mean()
diagnostics["mse_contribution_bp2"] = (
    1e8 * diagnostics["aggregate_mse_contribution"]
)
diagnostics["mse_share_pct"] = (
    100 * diagnostics["aggregate_mse_contribution"] / aggregate_mse
)
diagnostics["abs_mse_share_pct"] = diagnostics["mse_share_pct"].abs()
diagnostics = diagnostics.sort_values("abs_mse_share_pct", ascending=False)
display(diagnostics.drop(columns="aggregate_mse_contribution").head(25).round(3))


## 15. Latest estimate and output files


In [ ]:
target = final_mm.dropna().index[-1]
index = core_index.copy()
for t in final_mm.index[final_mm.index > core_index.index[-1]]:
    previous = t - pd.DateOffset(months=1)
    if previous in index.index:
        index.loc[t] = index.loc[previous] * (1 + final_mm.loc[t])
index = index.sort_index()

forecast_mm = final_mm.loc[target]
forecast_ann = 100 * ((1 + forecast_mm) ** 12 - 1)

historical_error_bp = 10000 * (
    bt.loc[EVALUATION_START:, "final_mm"] - bt.loc[EVALUATION_START:, "actual_mm"]
).dropna()
abs_error = historical_error_bp.abs()
bands = abs_error.quantile([0.50, 0.80, 0.90, 0.95])

print(f"Improved core PCE estimate for {target:%B %Y}")
print(f"  m/m                 : {100 * forecast_mm:5.3f}%")
print(f"  1-month annualized  : {forecast_ann:5.2f}%")
if target in index.index and target - pd.DateOffset(months=6) in index.index:
    print(f"  6-month annualized  : {100 * ((index.loc[target] / index.loc[target - pd.DateOffset(months=6)]) ** 2 - 1):5.2f}%")
if target in index.index and target - pd.DateOffset(months=12) in index.index:
    print(f"  12-month            : {100 * (index.loc[target] / index.loc[target - pd.DateOffset(months=12)] - 1):5.2f}%")
print(
    f"  empirical 90% error band: +/-{bands.loc[0.90]:.1f} bp m/m "
    "(current-vintage history)"
)
print(f"  hospital WPS ensemble weight: {hospital_wps_weight.loc[target]:.3f}")
print(f"  online bias adjustment: {-10000 * online_bias.fillna(0).loc[target]:+.2f} bp")

if target in bt.index and pd.notna(bt.loc[target, "actual_mm"]):
    print(f"  actual m/m          : {100 * bt.loc[target, 'actual_mm']:5.3f}%")
    print(f"  final error         : {10000 * (forecast_mm - bt.loc[target, 'actual_mm']):+.2f} bp")

detail = pd.DataFrame({
    "description": descriptions.reindex(core_lines).str.slice(0, 55),
    "source": {c: XWALK[c][0] for c in core_lines},
    "weight": core_weights.loc[last_bea],
    "proxy_estimate_pct": 100 * proxy_wps.loc[target],
    "component_ensemble_pct": 100 * component_ensemble.loc[target],
    "proxy_weight": alpha_wps.loc[target],
    "raw_fallback": raw_from_fallback.loc[target],
}).sort_values("weight", key=abs, ascending=False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
bt.round(8).to_csv(OUTPUT_DIR / "core_pce_nowcast_improved_backtest.csv")
metrics.round(6).to_csv(OUTPUT_DIR / "core_pce_nowcast_improved_metrics.csv")
detail.round(6).to_csv(OUTPUT_DIR / "core_pce_nowcast_improved_components_latest.csv")
diagnostics.round(8).to_csv(OUTPUT_DIR / "core_pce_nowcast_improved_diagnostics.csv")

print("\nWrote improved backtest, metrics, latest components, and diagnostics to outputs/")
display(detail.head(20).round(4))


## Interpretation and safeguards

- The main score begins in 2017, after 24 months of warm-up. Every adaptive weight,
  hospital-ensemble weight, and bias adjustment is shifted so month *t* uses only
  outcomes through *t-1*.
- This is still a **current-vintage** historical test. A vintage store of every BLS
  and BEA release is required before describing the metrics as true real-time
  performance.
- The detailed BEA partition and crosswalk should be reviewed after every annual
  NIPA update. Method changes should receive effective dates rather than being
  applied retroactively.
- The empirical uncertainty band is descriptive, not a guarantee. It should be
  widened after a vintage-correct backtest becomes available.
- `API Keys.txt` should remain untracked, or use `BLS_API_KEY`, `FRED_API_KEY`, and
  `BEA_API_KEY` environment variables.
